> Note: This notebook is an exercise workbook companion. Some cells intentionally contain `TODO` prompts or `NotImplementedError` placeholders for the reader to complete. For fully maintained runnable reference code, see `src/`, `tests/`, and the printed listings in the book.

# Chapter 2: Gradient Descent and Momentum

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vijaygwu/Optimization-for-Machine-Learning/blob/main/notebooks/ch02_gradient_descent.ipynb)

## Learning Objectives

By the end of this notebook, you will be able to:
1. Visualize loss surfaces in 3D and understand their geometry
2. Animate optimization trajectories on contour plots
3. Understand the impact of step size on convergence
4. Analyze convergence rates for different function types
5. Understand how condition number affects optimization difficulty

---

In [ ]:
# Setup and imports
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm, animation
from matplotlib.colors import LogNorm
import warnings
warnings.filterwarnings('ignore')

# Try to import ipywidgets
try:
    from ipywidgets import interact, interactive, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available. Install with: pip install ipywidgets")

# For Colab animation display
from IPython.display import HTML

np.random.seed(42)
print("Setup complete!")


## 1. 3D Loss Surface Visualization

Understanding the geometry of loss surfaces is crucial for intuition about optimization. Different loss surfaces have different properties:

- **Convex bowls**: Single global minimum, easy to optimize
- **Ravines**: Ill-conditioned, causes oscillation
- **Saddle points**: Flat regions that slow down optimization
- **Multiple minima**: Non-convex landscapes common in deep learning


In [ ]:
def plot_3d_surface(f, x_range=(-3, 3), y_range=(-3, 3), title="Loss Surface",
                    elev=30, azim=45, n_points=100):
    """
    Create a 3D visualization of a loss surface.

    Parameters:
    -----------
    f : callable
        Function f(x, y) -> z
    x_range, y_range : tuple
        Range for x and y axes
    title : str
        Plot title
    elev, azim : float
        Camera elevation and azimuth angles
    """
    fig = plt.figure(figsize=(14, 5))

    # Create mesh
    x = np.linspace(x_range[0], x_range[1], n_points)
    y = np.linspace(y_range[0], y_range[1], n_points)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)

    # 3D surface plot
    ax1 = fig.add_subplot(121, projection='3d')
    surf = ax1.plot_surface(X, Y, Z, cmap='viridis', alpha=0.8,
                            linewidth=0, antialiased=True)
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    ax1.set_zlabel('f(x,y)')
    ax1.set_title(f'{title} - 3D View')
    ax1.view_init(elev=elev, azim=azim)

    # Contour plot
    ax2 = fig.add_subplot(122)
    levels = 30
    try:
        cs = ax2.contour(X, Y, Z, levels=levels, cmap='viridis')
        ax2.clabel(cs, inline=True, fontsize=8)
    except:
        cs = ax2.contourf(X, Y, Z, levels=levels, cmap='viridis')

    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_title(f'{title} - Contour Plot')
    ax2.set_aspect('equal')

    plt.tight_layout()
    plt.show()

    return X, Y, Z

# 1. Simple quadratic (spherical bowl)
def quadratic_bowl(x, y):
    return x**2 + y**2

print("1. SPHERICAL BOWL: f(x,y) = x^2 + y^2")
print("   - Perfectly conditioned (condition number = 1)")
print("   - Single global minimum at (0, 0)")
plot_3d_surface(quadratic_bowl, title="Spherical Bowl")

# 2. Elongated quadratic (ravine)
def elongated_bowl(x, y):
    return x**2 + 50*y**2

print("\n2. ELONGATED BOWL (RAVINE): f(x,y) = x^2 + 50y^2")
print("   - Ill-conditioned (condition number = 50)")
print("   - Causes oscillation with vanilla GD")
plot_3d_surface(elongated_bowl, title="Elongated Bowl (Ravine)",
                x_range=(-5, 5), y_range=(-1, 1))

# 3. Rosenbrock function (banana valley)
def rosenbrock(x, y):
    return (1 - x)**2 + 100*(y - x**2)**2

print("\n3. ROSENBROCK FUNCTION: f(x,y) = (1-x)^2 + 100(y-x^2)^2")
print("   - Famous test function")
print("   - Global minimum at (1, 1)")
print("   - Narrow curved valley makes optimization difficult")
plot_3d_surface(rosenbrock, title="Rosenbrock (Banana Valley)",
                x_range=(-2, 2), y_range=(-1, 3))

# 4. Saddle point
def saddle(x, y):
    return x**2 - y**2

print("\n4. SADDLE POINT: f(x,y) = x^2 - y^2")
print("   - Minimum in x direction, maximum in y direction")
print("   - Gradient is zero at saddle point but it's not a minimum")
plot_3d_surface(saddle, title="Saddle Point")

# 5. Multiple local minima
def multimodal(x, y):
    return np.sin(x) * np.cos(y) + 0.1*(x**2 + y**2)

print("\n5. MULTIMODAL: f(x,y) = sin(x)cos(y) + 0.1(x^2 + y^2)")
print("   - Multiple local minima")
print("   - Quadratic term pulls towards origin")
plot_3d_surface(multimodal, title="Multimodal (Multiple Minima)",
                x_range=(-4, 4), y_range=(-4, 4))


## 2. Animated Optimization Trajectory

Let's visualize how gradient descent navigates different loss surfaces. We'll animate the optimization process on 2D contour plots.


In [ ]:
def gradient_descent_2d(f, grad_f, x0, lr=0.1, max_iter=100, tol=1e-8):
    """
    2D Gradient descent with history.

    Parameters:
    -----------
    f : callable
        Objective function f(x, y) -> scalar
    grad_f : callable
        Gradient function grad_f(x, y) -> (dx, dy)
    x0 : tuple
        Initial point (x, y)
    lr : float
        Learning rate
    max_iter : int
        Maximum iterations
    tol : float
        Convergence tolerance

    Returns:
    --------
    history : list of (x, y, f_val) tuples
    """
    x, y = x0
    history = [(x, y, f(x, y))]

    for i in range(max_iter):
        gx, gy = grad_f(x, y)
        x_new = x - lr * gx
        y_new = y - lr * gy

        history.append((x_new, y_new, f(x_new, y_new)))

        if np.sqrt((x_new - x)**2 + (y_new - y)**2) < tol:
            break

        x, y = x_new, y_new

    return history


def create_optimization_animation(f, grad_f, x0, lr, x_range, y_range,
                                  title="Gradient Descent", max_iter=100):
    """Create animation of gradient descent on contour plot."""

    # Run gradient descent
    history = gradient_descent_2d(f, grad_f, x0, lr=lr, max_iter=max_iter)
    path = np.array([(h[0], h[1]) for h in history])

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 8))

    # Create contour plot
    x = np.linspace(x_range[0], x_range[1], 200)
    y = np.linspace(y_range[0], y_range[1], 200)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)

    # Use log scale if range is large
    z_range = Z.max() - Z.min()
    if z_range > 100:
        levels = np.logspace(np.log10(max(Z.min(), 0.01)), np.log10(Z.max()), 30)
        cs = ax.contour(X, Y, Z, levels=levels, cmap='viridis')
    else:
        cs = ax.contour(X, Y, Z, levels=30, cmap='viridis')

    ax.clabel(cs, inline=True, fontsize=8)

    # Plot elements
    point, = ax.plot([], [], 'ro', markersize=12)
    line, = ax.plot([], [], 'r-', linewidth=2, alpha=0.7)

    # Mark start and end
    ax.plot(x0[0], x0[1], 'gs', markersize=15, label=f'Start {x0}')

    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim(x_range)
    ax.set_ylim(y_range)
    ax.legend(loc='upper right')

    def init():
        point.set_data([], [])
        line.set_data([], [])
        return point, line

    def animate(i):
        point.set_data([path[i, 0]], [path[i, 1]])
        line.set_data(path[:i+1, 0], path[:i+1, 1])
        ax.set_title(f'{title}\nIteration {i}, f = {history[i][2]:.6f}')
        return point, line

    anim = animation.FuncAnimation(fig, animate, init_func=init,
                                   frames=len(path), interval=50, blit=True)
    plt.close()
    return anim, history


# Define gradients for our test functions
def grad_quadratic(x, y):
    return 2*x, 2*y

def grad_elongated(x, y):
    return 2*x, 100*y

def grad_rosenbrock(x, y):
    dx = -2*(1 - x) - 400*x*(y - x**2)
    dy = 200*(y - x**2)
    return dx, dy

# Animation 1: Spherical bowl (easy)
print("Animation 1: Spherical Bowl")
print("=" * 50)
anim1, hist1 = create_optimization_animation(
    quadratic_bowl, grad_quadratic,
    x0=(2.5, 2.5), lr=0.2,
    x_range=(-3, 3), y_range=(-3, 3),
    title="Spherical Bowl (lr=0.2)"
)
HTML(anim1.to_jshtml())


In [ ]:
# Animation 2: Elongated bowl (ill-conditioned)
print("Animation 2: Elongated Bowl (Ill-conditioned)")
print("=" * 50)
anim2, hist2 = create_optimization_animation(
    elongated_bowl, grad_elongated,
    x0=(4, 0.5), lr=0.015,
    x_range=(-5, 5), y_range=(-1, 1),
    title="Elongated Bowl (lr=0.015)",
    max_iter=200
)
HTML(anim2.to_jshtml())


In [ ]:
# Animation 3: Rosenbrock function
print("Animation 3: Rosenbrock Function")
print("=" * 50)
anim3, hist3 = create_optimization_animation(
    rosenbrock, grad_rosenbrock,
    x0=(-1.5, 2.0), lr=0.001,
    x_range=(-2, 2), y_range=(-1, 3),
    title="Rosenbrock (lr=0.001)",
    max_iter=500
)
HTML(anim3.to_jshtml())


## 3. Step Size Experiments

The learning rate (step size) $\alpha$ is crucial:
- **Too small**: Very slow convergence, wasting computation
- **Too large**: Divergence or oscillation
- **Just right**: Fast, stable convergence

For a quadratic with Hessian eigenvalues $\lambda_{min}$ and $\lambda_{max}$:
- Convergence requires: $\alpha < 2/\lambda_{max}$
- Optimal rate when: $\alpha = 2/(\lambda_{min} + \lambda_{max})$


In [ ]:
def step_size_experiment():
    """Demonstrate effects of different step sizes."""

    # Use simple quadratic: f(x,y) = x^2 + 10y^2
    # Hessian eigenvalues: 2 and 20
    # Convergence requires lr < 2/20 = 0.1
    # Optimal lr = 2/(2+20) = 0.091

    def f(x, y):
        return x**2 + 10*y**2

    def grad_f(x, y):
        return 2*x, 20*y

    x0 = (4, 2)

    # Different learning rates
    learning_rates = {
        'Too small (0.01)': 0.01,
        'Good (0.05)': 0.05,
        'Optimal (0.091)': 0.091,
        'Large (0.095)': 0.095,
        'Too large (0.11)': 0.11,  # Should diverge
    }

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    # Contour plot setup
    x = np.linspace(-5, 5, 100)
    y = np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)

    for ax, (name, lr) in zip(axes[:-1], learning_rates.items()):
        # Run GD
        history = gradient_descent_2d(f, grad_f, x0, lr=lr, max_iter=100)
        path = np.array([(h[0], h[1]) for h in history])
        losses = [h[2] for h in history]

        # Contour
        cs = ax.contour(X, Y, Z, levels=20, cmap='Blues', alpha=0.5)

        # Trajectory
        if len(path) > 1:
            # Clip path to visible range
            path_clipped = path[(np.abs(path[:, 0]) < 5) & (np.abs(path[:, 1]) < 3)]
            if len(path_clipped) > 1:
                ax.plot(path_clipped[:, 0], path_clipped[:, 1], 'r.-', markersize=4, linewidth=1)

        ax.plot(x0[0], x0[1], 'gs', markersize=10)
        ax.plot(0, 0, 'k*', markersize=15)

        # Check convergence
        final_loss = losses[-1]
        if final_loss < 0.01:
            status = f"Converged! ({len(history)} iters)"
        elif final_loss > losses[0]:
            status = "DIVERGED!"
        else:
            status = f"Slow... (loss={final_loss:.2f})"

        ax.set_title(f'{name}\n{status}')
        ax.set_xlim(-5, 5)
        ax.set_ylim(-3, 3)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.grid(True, alpha=0.3)

    # Loss curves comparison
    ax = axes[-1]
    for name, lr in learning_rates.items():
        history = gradient_descent_2d(f, grad_f, x0, lr=lr, max_iter=100)
        losses = [h[2] for h in history]
        # Clip for visualization
        losses = np.clip(losses, 0, 200)
        ax.semilogy(losses[:50], linewidth=2, label=f'lr={lr}')

    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title('Loss Curves Comparison')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Theory explanation
    print("\nTheoretical Analysis:")
    print("=" * 50)
    print("Function: f(x,y) = x^2 + 10y^2")
    print("Hessian eigenvalues: lambda_min=2, lambda_max=20")
    print(f"Condition number: {20/2}")
    print(f"\nStability condition: lr < 2/lambda_max = {2/20}")
    print(f"Optimal learning rate: 2/(lambda_min + lambda_max) = {2/22:.4f}")

step_size_experiment()


In [ ]:
def interactive_step_size(lr=0.05, condition_number=10):
    """Interactive exploration of step size effects."""

    # f(x,y) = x^2 + kappa * y^2
    kappa = condition_number

    def f(x, y):
        return x**2 + kappa * y**2

    def grad_f(x, y):
        return 2*x, 2*kappa*y

    x0 = (3, 1)

    # Theoretical bounds
    lambda_max = 2 * kappa
    stable_lr = 2 / lambda_max
    optimal_lr = 2 / (2 + 2*kappa)

    # Run GD
    history = gradient_descent_2d(f, grad_f, x0, lr=lr, max_iter=200)
    path = np.array([(h[0], h[1]) for h in history])
    losses = [h[2] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Trajectory plot
    ax = axes[0]
    x = np.linspace(-4, 4, 100)
    y = np.linspace(-2, 2, 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)

    ax.contour(X, Y, Z, levels=20, cmap='Blues')

    # Clip trajectory
    mask = (np.abs(path[:, 0]) < 4) & (np.abs(path[:, 1]) < 2)
    path_show = path[mask] if mask.sum() > 0 else path[:1]
    ax.plot(path_show[:, 0], path_show[:, 1], 'r.-', markersize=4)
    ax.plot(x0[0], x0[1], 'gs', markersize=12, label='Start')
    ax.plot(0, 0, 'k*', markersize=15, label='Optimum')

    # Status
    if losses[-1] < 0.001:
        status = f"CONVERGED in {len(history)} iterations"
        color = 'green'
    elif losses[-1] > losses[0]:
        status = "DIVERGED!"
        color = 'red'
    else:
        status = f"In progress (loss={losses[-1]:.4f})"
        color = 'orange'

    ax.set_title(f'Trajectory (kappa={kappa})\n{status}', color=color)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-2, 2)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Loss curve
    ax = axes[1]
    ax.semilogy(np.clip(losses, 1e-10, 1e6), 'b-', linewidth=2)
    ax.axhline(y=0.001, color='green', linestyle='--', label='Convergence threshold')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss (log scale)')
    ax.set_title(f'Loss Curve\nlr={lr}, stable_lr<{stable_lr:.4f}, optimal={optimal_lr:.4f}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f"Condition number: {kappa}")
    print(f"Learning rate: {lr}")
    print(f"Stability requires: lr < {stable_lr:.4f}")
    print(f"Optimal lr: {optimal_lr:.4f}")
    print(f"Final loss: {losses[-1]:.6f}")

if WIDGETS_AVAILABLE:
    interact(interactive_step_size,
             lr=FloatSlider(min=0.001, max=0.2, step=0.001, value=0.05, description='Learning Rate'),
             condition_number=IntSlider(min=1, max=100, step=1, value=10, description='Cond. Number'))
else:
    for lr in [0.01, 0.05, 0.09]:
        interactive_step_size(lr=lr, condition_number=10)


## 4. Convergence Rate Visualization

Convergence rates describe how quickly the optimization error decreases:

| Function Class | Rate | Bound |
|---------------|------|-------|
| Convex | $O(1/k)$ | $f(x_k) - f^* \leq \frac{\|x_0 - x^*\|^2}{2\alpha k}$ |
| Strongly convex | $O((1-\mu/L)^k)$ | Linear convergence |
| Smooth + SC | $O(\rho^k)$ | $\rho = \frac{\kappa - 1}{\kappa + 1}$ |

Where $\kappa = L/\mu$ is the condition number.


In [ ]:
def visualize_convergence_rates():
    """Visualize theoretical vs empirical convergence rates."""

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # 1. Convex function: f(x) = |x|^1.5 + 0.1*x^2
    # (This is convex but not strongly convex near origin)
    def convex_func(x):
        return np.abs(x)**1.5 + 0.1*x**2

    def grad_convex(x):
        return 1.5 * np.sign(x) * np.abs(x)**0.5 + 0.2*x

    # GD for 1D
    x = 5.0
    lr = 0.1
    x_hist = [x]
    f_hist = [convex_func(x)]

    for _ in range(500):
        x = x - lr * grad_convex(x)
        x_hist.append(x)
        f_hist.append(convex_func(x))

    ax = axes[0, 0]
    f_star = convex_func(0)
    errors = np.array(f_hist) - f_star
    ax.loglog(range(1, len(errors)+1), errors, 'b-', linewidth=2, label='Empirical')
    # O(1/k) reference
    k = np.arange(1, len(errors)+1)
    ax.loglog(k, errors[0]/k, 'r--', linewidth=2, label='O(1/k) theory')
    ax.set_xlabel('Iteration k')
    ax.set_ylabel('f(x_k) - f*')
    ax.set_title('Convex (not strongly): O(1/k) rate')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 2. Strongly convex: f(x) = x^2 (mu = 2, L = 2)
    def sc_func(x):
        return x**2

    def grad_sc(x):
        return 2*x

    x = 5.0
    lr = 0.5  # = 1/L
    x_hist_sc = [x]
    f_hist_sc = [sc_func(x)]

    for _ in range(50):
        x = x - lr * grad_sc(x)
        x_hist_sc.append(x)
        f_hist_sc.append(sc_func(x))

    ax = axes[0, 1]
    f_star = 0
    errors_sc = np.array(f_hist_sc) - f_star
    ax.semilogy(errors_sc, 'b-', linewidth=2, label='Empirical')
    # Linear convergence reference
    kappa = 1  # L/mu = 1 for f(x) = x^2
    rho = (1 - lr * 2)**2  # (1 - alpha*mu)^2 for quadratic
    k = np.arange(len(errors_sc))
    ax.semilogy(errors_sc[0] * rho**k, 'r--', linewidth=2, label=f'Linear rate rho={rho:.2f}')
    ax.set_xlabel('Iteration k')
    ax.set_ylabel('f(x_k) - f* (log)')
    ax.set_title('Strongly Convex: Linear rate')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 3. Effect of condition number
    ax = axes[1, 0]

    condition_numbers = [1, 2, 5, 10, 50, 100]
    colors = plt.cm.viridis(np.linspace(0, 1, len(condition_numbers)))

    for kappa, color in zip(condition_numbers, colors):
        # f(x,y) = x^2 + kappa*y^2
        def f_kappa(x, y, k=kappa):
            return x**2 + k*y**2
        def grad_kappa(x, y, k=kappa):
            return 2*x, 2*k*y

        # Optimal learning rate
        L = 2*kappa
        mu = 2
        lr = 2 / (L + mu)

        history = gradient_descent_2d(
            lambda x, y: f_kappa(x, y),
            lambda x, y: grad_kappa(x, y),
            (3, 1), lr=lr, max_iter=200
        )
        losses = [h[2] for h in history]
        ax.semilogy(losses, color=color, linewidth=2, label=f'kappa={kappa}')

    ax.set_xlabel('Iteration')
    ax.set_ylabel('f(x) - f* (log)')
    ax.set_title('Condition Number Effect on Convergence')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # 4. Convergence rate vs condition number
    ax = axes[1, 1]

    kappas = np.array([1, 2, 5, 10, 20, 50, 100, 200, 500])
    theoretical_rates = (kappas - 1) / (kappas + 1)

    # Empirical rates
    empirical_rates = []
    for kappa in kappas:
        L = 2*kappa
        mu = 2
        lr = 2 / (L + mu)

        history = gradient_descent_2d(
            lambda x, y, k=kappa: x**2 + k*y**2,
            lambda x, y, k=kappa: (2*x, 2*k*y),
            (3, 1), lr=lr, max_iter=100
        )
        losses = [h[2] for h in history]

        # Estimate rate from last iterations
        if len(losses) > 10 and losses[-1] > 1e-15:
            rate = (losses[-1] / losses[-11]) ** 0.1  # 10th root
        else:
            rate = 0
        empirical_rates.append(rate)

    ax.semilogx(kappas, theoretical_rates, 'b-o', linewidth=2, markersize=8,
                label='Theoretical (kappa-1)/(kappa+1)')
    ax.semilogx(kappas, empirical_rates, 'r--s', linewidth=2, markersize=8,
                label='Empirical')
    ax.set_xlabel('Condition Number (kappa)')
    ax.set_ylabel('Convergence Rate (rho)')
    ax.set_title('Convergence Rate vs Condition Number\n(higher rate = slower convergence)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

visualize_convergence_rates()


## 5. Condition Number Effects on Convergence

The **condition number** $\kappa = \frac{L}{\mu}$ measures the "difficulty" of optimization:

- $\kappa = 1$: Spherical level sets, easy optimization
- $\kappa >> 1$: Elongated level sets, difficult optimization

For quadratic $f(x) = \frac{1}{2}x^T H x$:
- $\kappa = \lambda_{max}(H) / \lambda_{min}(H)$
- Convergence rate: $\rho = \frac{\kappa - 1}{\kappa + 1}$
- Number of iterations to reduce error by factor $\epsilon$: $O(\kappa \log(1/\epsilon))$


In [ ]:
def condition_number_visualization():
    """Visualize how condition number affects optimization."""

    fig, axes = plt.subplots(2, 4, figsize=(16, 8))

    condition_numbers = [1, 2, 10, 50]

    for i, kappa in enumerate(condition_numbers):
        # Function with this condition number
        def f(x, y, k=kappa):
            return 0.5 * (x**2 + k * y**2)

        def grad_f(x, y, k=kappa):
            return x, k*y

        # Top row: contour plots with GD trajectory
        ax = axes[0, i]

        x = np.linspace(-3, 3, 100)
        y = np.linspace(-3, 3, 100)
        X, Y = np.meshgrid(x, y)
        Z = f(X, Y)

        ax.contour(X, Y, Z, levels=15, cmap='Blues')

        # GD trajectory
        L = kappa
        mu = 1
        lr = 2 / (L + mu)

        history = gradient_descent_2d(f, grad_f, (2.5, 2.5), lr=lr, max_iter=100)
        path = np.array([(h[0], h[1]) for h in history])

        ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=3, linewidth=1)
        ax.plot(2.5, 2.5, 'gs', markersize=8)
        ax.plot(0, 0, 'k*', markersize=12)

        ax.set_title(f'kappa = {kappa}')
        ax.set_xlim(-3, 3)
        ax.set_ylim(-3, 3)
        ax.set_aspect('equal')
        ax.set_xlabel('x')
        ax.set_ylabel('y')

        # Bottom row: convergence curves
        ax = axes[1, i]
        losses = [h[2] for h in history]
        ax.semilogy(losses, 'b-', linewidth=2)
        ax.axhline(y=1e-10, color='r', linestyle='--', alpha=0.5)
        ax.set_xlabel('Iteration')
        ax.set_ylabel('f(x)')
        ax.set_title(f'Convergence (kappa={kappa})\n{len(history)} iterations')
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Summary table
    print("\nSummary: Iterations to reach f(x) < 1e-10")
    print("=" * 50)
    print(f"{'Condition Number':<20} {'Iterations':<15} {'Rate rho':<15}")
    print("-" * 50)
    for kappa in condition_numbers:
        L = kappa
        mu = 1
        lr = 2 / (L + mu)

        history = gradient_descent_2d(
            lambda x, y, k=kappa: 0.5 * (x**2 + k * y**2),
            lambda x, y, k=kappa: (x, k*y),
            (2.5, 2.5), lr=lr, max_iter=1000
        )

        # Count iterations to convergence
        for i, h in enumerate(history):
            if h[2] < 1e-10:
                break
        iters = i

        rho = (kappa - 1) / (kappa + 1)
        print(f"{kappa:<20} {iters:<15} {rho:<15.4f}")

condition_number_visualization()


In [ ]:
def interactive_condition_number(kappa=10, lr_factor=1.0):
    """
    Interactive exploration of condition number effects.

    Parameters:
    -----------
    kappa : int
        Condition number
    lr_factor : float
        Multiplier for optimal learning rate (1.0 = optimal)
    """

    def f(x, y):
        return 0.5 * (x**2 + kappa * y**2)

    def grad_f(x, y):
        return x, kappa*y

    # Learning rate
    L = kappa
    mu = 1
    optimal_lr = 2 / (L + mu)
    lr = optimal_lr * lr_factor

    # Run optimization
    x0 = (2.5, 2.5)
    history = gradient_descent_2d(f, grad_f, x0, lr=lr, max_iter=500)
    path = np.array([(h[0], h[1]) for h in history])
    losses = [h[2] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Contour with trajectory
    ax = axes[0]
    x = np.linspace(-3, 3, 100)
    y = np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)

    ax.contour(X, Y, Z, levels=15, cmap='Blues')
    ax.plot(path[:, 0], path[:, 1], 'r.-', markersize=3, linewidth=1)
    ax.plot(x0[0], x0[1], 'gs', markersize=10, label='Start')
    ax.plot(0, 0, 'k*', markersize=15, label='Optimum')

    ax.set_title(f'Condition Number = {kappa}\nLR = {lr:.4f} ({lr_factor}x optimal)')
    ax.set_xlim(-3, 3)
    ax.set_ylim(-3, 3)
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Convergence
    ax = axes[1]
    ax.semilogy(losses, 'b-', linewidth=2)

    # Theoretical rate
    if lr <= 2/L:  # Stable
        rho = max(abs(1 - lr*mu), abs(1 - lr*L))
        theory = [losses[0] * rho**(2*k) for k in range(len(losses))]
        ax.semilogy(theory, 'r--', alpha=0.7, label=f'Theory (rho={rho:.3f})')

    ax.axhline(y=1e-10, color='green', linestyle='--', alpha=0.5, label='Convergence')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('f(x) (log scale)')
    ax.set_title(f'Convergence\nFinal loss: {losses[-1]:.2e}')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Info
    print(f"Condition number kappa: {kappa}")
    print(f"Optimal LR: {optimal_lr:.4f}")
    print(f"Used LR: {lr:.4f}")
    print(f"Theoretical convergence rate: {(kappa-1)/(kappa+1):.4f}")
    print(f"Iterations: {len(history)}")
    print(f"Final loss: {losses[-1]:.2e}")

if WIDGETS_AVAILABLE:
    interact(interactive_condition_number,
             kappa=IntSlider(min=1, max=100, step=1, value=10, description='Cond. Number'),
             lr_factor=FloatSlider(min=0.1, max=2.0, step=0.1, value=1.0, description='LR Factor'))
else:
    for k in [1, 10, 50]:
        interactive_condition_number(kappa=k, lr_factor=1.0)


---

## Interview Questions

### Question 1: Learning Rate Selection
**Q: You're training a neural network and notice the loss is oscillating wildly. What's likely causing this and how would you fix it?**

<details>
<summary>Click for Answer</summary>

**Likely Cause:** Learning rate is too large.

**Diagnosis:**
- Oscillating loss suggests overshooting the minimum
- Each gradient step jumps past the optimal point
- Can also happen with high variance gradients (small batch)

**Solutions:**

1. **Reduce learning rate**: Try reducing by 2-10x
2. **Learning rate scheduling**: Start high, decay over time
3. **Warmup**: Start with small LR, gradually increase
4. **Adaptive methods**: Use Adam/AdaGrad which adapt per-parameter
5. **Gradient clipping**: Limit gradient magnitude to prevent large steps
6. **Increase batch size**: Reduces gradient variance

**Investigation Steps:**
1. Plot loss curve - look for oscillation pattern
2. Try 10x smaller LR - see if oscillation stops
3. Check gradient norms - large gradients suggest instability
4. Monitor parameter updates - should be small relative to parameter magnitudes
</details>

### Question 2: Condition Number
**Q: What is the condition number and why does it matter for optimization? How can you address ill-conditioning?**

<details>
<summary>Click for Answer</summary>

**Condition Number Definition:**
$$\kappa = \frac{\lambda_{max}}{\lambda_{min}}$$

Ratio of largest to smallest eigenvalues of the Hessian (for quadratics) or the smoothness constant L to strong convexity constant mu.

**Why It Matters:**
- $\kappa = 1$: Spherical level sets, same curvature in all directions
- $\kappa >> 1$: Elongated "ravine" - very different curvatures
- Convergence rate: $(\kappa - 1)/(\kappa + 1)$
- Iterations needed: $O(\kappa \log(1/\epsilon))$

**High Condition Number Problems:**
- Slow convergence along "long" directions
- Oscillation along "short" directions
- Optimal learning rate is tiny

**Solutions:**

1. **Preconditioning**: Transform problem to have lower condition number
   - Newton's method: Use Hessian inverse
   - Natural gradient: Use Fisher information matrix

2. **Adaptive methods**:
   - Adam/RMSprop adapt to local curvature
   - Effectively precondition using gradient history

3. **Normalization**:
   - Feature normalization (mean=0, std=1)
   - Batch normalization in deep learning
   - Layer normalization

4. **Momentum**: Helps traverse ravines faster

5. **Second-order methods**: Use curvature information (expensive)
</details>

### Question 3: Convergence Guarantees
**Q: What convergence rate does gradient descent achieve for smooth convex functions? What about strongly convex functions? How do you improve these rates?**

<details>
<summary>Click for Answer</summary>

**Smooth Convex Functions:**
- Rate: $O(1/k)$ (sublinear)
- After $k$ iterations: $f(x_k) - f^* \leq O(1/k)$
- Need $O(1/\epsilon)$ iterations for $\epsilon$-accuracy

**Strongly Convex Functions:**
- Rate: $O(\rho^k)$ where $\rho = (\kappa-1)/(\kappa+1)$ (linear)
- Error decreases exponentially
- Need $O(\kappa \log(1/\epsilon))$ iterations

**Improving Convergence:**

1. **Acceleration (Nesterov):**
   - Smooth convex: $O(1/k)$ -> $O(1/k^2)$
   - Strongly convex: $O(\kappa \log(1/\epsilon))$ -> $O(\sqrt{\kappa} \log(1/\epsilon))$
   - Uses momentum with specific coefficient

2. **Adaptive methods:**
   - Effective preconditioning reduces effective condition number
   - Adam, AdaGrad, RMSprop

3. **Second-order methods:**
   - Newton: Quadratic convergence near optimum
   - Quasi-Newton (BFGS, L-BFGS): Approximate Hessian

4. **Variance reduction (for SGD):**
   - SVRG, SAGA achieve linear convergence for finite sums
   - Match full-batch GD rate with stochastic updates
</details>

---


## Exercises

### Exercise 1: Implement Nesterov Momentum
Implement gradient descent with Nesterov momentum (accelerated gradient descent):

$$y_{t+1} = x_t - \alpha \nabla f(x_t)$$
$$x_{t+1} = y_{t+1} + \frac{t}{t+3}(y_{t+1} - y_t)$$

Compare convergence with vanilla GD on an ill-conditioned quadratic.


In [ ]:
# Exercise 1: Your solution here

def nesterov_gd(f, grad_f, x0, lr=0.1, max_iter=100):
    """
    Nesterov accelerated gradient descent.

    TODO: Implement this

    Hint: Keep track of both y (gradient step) and x (momentum step)
    """
    pass

# Test on f(x,y) = x^2 + 50*y^2
# Compare vanilla GD vs Nesterov


### Exercise 2: Line Search
Implement gradient descent with backtracking line search (Armijo condition):

Find step size $\alpha$ such that:
$$f(x - \alpha \nabla f(x)) \leq f(x) - c \alpha \|\nabla f(x)\|^2$$

Start with $\alpha = 1$ and reduce by factor $\beta$ until condition is satisfied.


In [ ]:
# Exercise 2: Your solution here

def gd_line_search(f, grad_f, x0, c=0.5, beta=0.5, max_iter=100):
    """
    Gradient descent with Armijo backtracking line search.

    TODO: Implement this

    Parameters:
    -----------
    c : float
        Armijo condition parameter (typically 0.5)
    beta : float
        Step size reduction factor (typically 0.5)
    """
    pass

# Test on Rosenbrock function
# Compare fixed LR vs line search


### Exercise 3: Saddle Point Escape
Gradient descent can get stuck at saddle points. Implement a simple saddle-point escape mechanism by adding noise when the gradient is small but the function value is high.

Test on the function $f(x,y) = x^2 - y^2$ which has a saddle at origin.


In [ ]:
# Exercise 3: Your solution here

def gd_with_escape(f, grad_f, x0, lr=0.1, noise_scale=0.1, grad_threshold=0.01, max_iter=100):
    """
    Gradient descent with saddle point escape.

    When gradient norm is below threshold, add random noise.

    TODO: Implement this
    """
    pass

# Test: Start near saddle point (0, 0) of f(x,y) = x^2 - y^2
# The algorithm should escape and move towards x=0, y->infinity (or -infinity)
